# 00 — First Look at the DataCo Supply Chain Data
**Goal:** understand the raw data before changing anything.
**Outputs:** column notes, grain, leakage list, questions list.
**Rule:** read only, no cleaning in this notebook.

In [2]:
import sys

print(sys.executable)

d:\DataCo Supply Chain Project\supply-chain-data-science\.venv\Scripts\python.exe


In [3]:
from pathlib import Path
import pandas as pd

In [4]:
ROOT = Path.cwd().parent  # notebook lives in notebooks/, so go up one level
RAW = ROOT / "data" / "raw"

orders = pd.read_csv(RAW / "DataCoSupplyChainDataset.csv", encoding="latin-1")
logs = pd.read_csv(RAW / "tokenized_access_logs.csv")
desc = pd.read_csv(RAW / "DescriptionDataCoSupplyChain.csv", encoding="latin-1")

for name, df in {"orders": orders, "logs": logs, "desc": desc}.items():
    print(f"{name:7} {df.shape}")

orders  (180519, 53)
logs    (469977, 8)
desc    (52, 2)


In [5]:
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", None)

In [6]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  str    
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  str    
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  str    
 9   Customer City                  180519 non-null  str    
 10  Customer Country               180519 non-null  str    
 11  Customer Email                 180519 non-null  str    
 12  Customer Fname                 180519 non

In [17]:
orders.head(10).T

,0,1,2,3,4,5,6,7,8,9
Type,DEBIT,TRANSFER,CASH,DEBIT,PAYMENT,TRANSFER,DEBIT,TRANSFER,CASH,CASH
Days for shipping (real),3,5,4,3,2,6,2,2,3,2
Days for shipment (scheduled),4,4,4,4,4,4,1,1,2,1
Benefit per order,91.25,-249.089996,-247.779999,22.860001,134.210007,18.58,95.18,68.43,133.720001,132.149994
Sales per customer,314.640015,311.359985,309.720001,304.809998,298.25,294.980011,288.420013,285.140015,278.589996,275.309998
Delivery Status,Advance shipping,Late delivery,Shipping on time,Advance shipping,Advance shipping,Shipping canceled,Late delivery,Late delivery,Late delivery,Late delivery
Late_delivery_risk,0,1,0,0,0,0,1,1,1,1
Category Id,73,73,73,73,73,73,73,73,73,73
Category Name,Sporting Goods,Sporting Goods,Sporting Goods,Sporting Goods,Sporting Goods,Sporting Goods,Sporting Goods,Sporting Goods,Sporting Goods,Sporting Goods
Customer City,Caguas,Caguas,San Jose,Los Angeles,Caguas,Tonawanda,Caguas,Miami,Caguas,San Ramon


In [15]:
nulls = orders.isna().sum()
nulls[nulls > 0].sort_values(ascending=False)

Product Description    180519
Order Zipcode          155679
Customer Lname              8
Customer Zipcode            3
dtype: int64

In [16]:
orders.nunique().sort_values()

Product Description                   0
Customer Password                     1
Product Status                        1
Customer Email                        1
Late_delivery_risk                    2
Customer Country                      2
Customer Segment                      3
Delivery Status                       4
Days for shipment (scheduled)         4
Shipping Mode                         4
Type                                  4
Market                                5
Order Item Quantity                   5
Days for shipping (real)              7
Order Status                          9
Department Id                        11
Department Name                      11
Order Item Discount Rate             18
Order Region                         23
Customer State                       46
Category Name                        50
Product Category Id                  51
Category Id                          51
Order Item Product Price             75
Product Price                        75


In [40]:
orders["Order Status"].nunique()

9

In [22]:
orders.groupby("Market")["Order Zipcode"].apply(lambda s: s.isna().mean())

Market
Africa          1.000000
Europe          1.000000
LATAM           1.000000
Pacific Asia    1.000000
USCA            0.037172
Name: Order Zipcode, dtype: float64

In [28]:
zo = orders["Order Zipcode"]

zo.dropna(axis=0, inplace=True)
zo

55        99301.0
56        90049.0
57        94110.0
58        66212.0
183       77041.0
           ...   
179402    94110.0
179403    94122.0
179404    42420.0
179405    94122.0
179406    19143.0
Name: Order Zipcode, Length: 24840, dtype: float64

In [18]:
logs.info()

<class 'pandas.DataFrame'>
RangeIndex: 469977 entries, 0 to 469976
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   Product     469977 non-null  str  
 1   Category    469977 non-null  str  
 2   Date        469977 non-null  str  
 3   Month       469977 non-null  str  
 4   Hour        469977 non-null  int64
 5   Department  469977 non-null  str  
 6   ip          469977 non-null  str  
 7   url         469977 non-null  str  
dtypes: int64(1), str(7)
memory usage: 114.9 MB


In [ ]:
logs.head(3)

,Product,Category,Date,Month,Hour,Department,ip,url
0,adidas Brazuca 2017 Official Match Ball,baseball & softball,9/1/2017 6:00,Sep,6,fitness,37.97.182.65,/department/fitness/category/baseball%20&%20softball/product/adidas%20Brazuc...
1,The North Face Women's Recon Backpack,hunting & shooting,9/1/2017 6:00,Sep,6,fan shop,206.56.112.1,/department/fan%20shop/category/hunting%20&%20shooting/product/The%20North%2...
2,adidas Kids' RG III Mid Football Cleat,featured shops,9/1/2017 6:00,Sep,6,apparel,215.143.180.0,/department/apparel/category/featured%20shops/product/adidas%20Kids'%20RG%20...


In [21]:
logs.nunique().sort_values()

Month              5
Department         6
Hour              24
Category          33
Product           76
url              152
ip              3340
Date          160815
dtype: int64

In [29]:
print(len(orders))
print(orders["Order Id"].nunique())
print(orders["Order Item Id"].nunique())

180519
65752
180519


In [31]:
multi = orders["Order Id"].value_counts()
example = multi[multi > 1].index[0]
orders[orders["Order Id"] == example]

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
49,PAYMENT,2,2,-122.730003,79.180000,Shipping on time,0,29,Shop By Sport,Caguas,Puerto Rico,XXXXXXXXX,Mary,4741,Smith,XXXXXXXXX,Home Office,PR,9731 Honey Fox Towers,725.0,5,Golf,18.235573,-66.370613,Pacific Asia,Bursa,Turquía,4741,10/25/2016 14:39,45461,627,0.80,0.01,113598,39.990002,-1.55,2,79.980003,79.180000,-122.730003,West Asia,Bursa,PENDING_PAYMENT,NaN,627,29,NaN,http://images.acmesports.sports/Under+Armour+Girls%27+Toddler+Spine+Surge+Ru...,Under Armour Girls' Toddler Spine Surge Runni,39.990002,0,10/27/2016 14:39,Second Class
515,PAYMENT,2,2,50.840000,164.000000,Shipping on time,0,24,Women's Apparel,Caguas,Puerto Rico,XXXXXXXXX,Mary,4741,Smith,XXXXXXXXX,Home Office,PR,9731 Honey Fox Towers,725.0,5,Golf,18.235573,-66.370613,Pacific Asia,Bursa,Turquía,4741,10/25/2016 14:39,45461,502,36.00,0.18,113596,50.000000,0.31,4,200.000000,164.000000,50.840000,West Asia,Bursa,PENDING_PAYMENT,NaN,502,24,NaN,http://images.acmesports.sports/Nike+Men%27s+Dri-FIT+Victory+Golf+Polo,Nike Men's Dri-FIT Victory Golf Polo,50.000000,0,10/27/2016 14:39,Second Class
112661,PAYMENT,2,2,-80.010002,119.949997,Shipping on time,0,46,Indoor/Outdoor Games,Caguas,Puerto Rico,XXXXXXXXX,Mary,4741,Smith,XXXXXXXXX,Home Office,PR,9731 Honey Fox Towers,725.0,7,Fan Shop,18.235573,-66.370613,Pacific Asia,Bursa,Turquía,4741,10/25/2016 14:39,45461,1014,29.99,0.20,113600,49.980000,-0.67,3,149.940002,119.949997,-80.010002,West Asia,Bursa,PENDING_PAYMENT,NaN,1014,46,NaN,http://images.acmesports.sports/O%27Brien+Men%27s+Neoprene+Life+Vest,O'Brien Men's Neoprene Life Vest,49.980000,0,10/27/2016 14:39,Second Class
179242,PAYMENT,2,2,47.770000,97.489998,Shipping on time,0,18,Men's Footwear,Caguas,Puerto Rico,XXXXXXXXX,Mary,4741,Smith,XXXXXXXXX,Home Office,PR,9731 Honey Fox Towers,725.0,4,Apparel,18.235573,-66.370613,Pacific Asia,Bursa,Turquía,4741,10/25/2016 14:39,45461,403,32.50,0.25,113597,129.990005,0.49,1,129.990005,97.489998,47.770000,West Asia,Bursa,PENDING_PAYMENT,NaN,403,18,NaN,http://images.acmesports.sports/Nike+Men%27s+CJ+Elite+2+TD+Football+Cleat,Nike Men's CJ Elite 2 TD Football Cleat,129.990005,0,10/27/2016 14:39,Second Class
179305,PAYMENT,2,2,16.000000,159.979996,Shipping on time,0,9,Cardio Equipment,Caguas,Puerto Rico,XXXXXXXXX,Mary,4741,Smith,XXXXXXXXX,Home Office,PR,9731 Honey Fox Towers,725.0,3,Footwear,18.235573,-66.370613,Pacific Asia,Bursa,Turquía,4741,10/25/2016 14:39,45461,191,40.00,0.20,113599,99.989998,0.10,2,199.979996,159.979996,16.000000,West Asia,Bursa,PENDING_PAYMENT,NaN,191,9,NaN,http://images.acmesports.sports/Nike+Men%27s+Free+5.0%2B+Running+Shoe,Nike Men's Free 5.0+ Running Shoe,99.989998,0,10/27/2016 14:39,Second Class


## Notes for cleaning (Step 3)

drop all of product description and order zipcode

fill customer lname and customer zipcode

drop all of 

Product Description                   
Customer Password                     
Product Status                        
Customer Email                        


## yes/no ##

Late_delivery_risk    2



Order Zipcode	100% null in Africa, Europe, LATAM, Pacific Asia; 3.7% null in USCA	 so drop it only record for USCA